In [1]:

# =============================================================================
# CELL 1 – IMPORTS, DARK THEME, AND HELPERS
# =============================================================================

import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# -------------------- Dark Theme --------------------
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 {
        color: #4fc3f7 !important;
        border-bottom: 2px solid #3498db !important;
        padding-bottom: 4px;
    }
    b, strong { color: #f48fb1 !important; }
    .highlight {
        background-color: #2d2d2d !important;
        padding: 10px;
        border-left: 4px solid #3498db;
        margin: 4px 0;
        color: #d4d4d4;
    }
    code {
        background-color: #333 !important;
        color: #ffcc80 !important;
        padding: 2px 4px;
        border-radius: 4px;
    }
    .dataframe {
        background-color: #2d2d2d !important;
        color: #d4d4d4 !important;
    }
</style>
"""))

# -------------------- Verbosity Flags --------------------
SHOW_VERBOSE = True
SHOW_INFO = True
SHOW_CRITICAL = True
SHOW_DEBUG = True
# For demonstration, we keep them False to reduce output; you can set True as needed.


# -------------------- Import Extraction Module --------------------
import Extraction6_ as Extraction6
importlib.reload(Extraction6)

# -------------------- Helper Functions --------------------
def display_title(title: str):
    display(HTML(f"<h2>{title}</h2>"))

def display_info(message: str):
    display(HTML(f"<div class='highlight'>{message}</div>"))

def load_experiment_results(exp_root: Path) -> pd.DataFrame:
    """
    Load all extraction results from the experiment directory, even if the
    run was interrupted. It reads per-dataset metadata files.
    """
    records = []
    exp_root = Path(exp_root)
    if not exp_root.exists():
        return pd.DataFrame()

    # Search for all extraction.json files under models/*/datasets/*/
    for meta_file in exp_root.rglob("extraction.json"):
        try:
            with open(meta_file, "r") as f:
                meta = json.load(f)
            # Extract relevant fields
            record = {
                "experiment_id": meta.get("experiment_id"),
                "model": meta.get("model", {}).get("name"),
                "dataset": meta.get("dataset", {}).get("name"),
                "status": meta.get("status"),
                "completed_samples": meta.get("performance", {}).get("completed_samples"),
                "total_samples": meta.get("dataset", {}).get("samples"),
                "batch_size": meta.get("extraction", {}).get("batch_size"),
                "pooling": meta.get("extraction", {}).get("pooling"),
                "max_length": meta.get("extraction", {}).get("max_length"),
                "samples_per_second": meta.get("performance", {}).get("samples_per_second"),
                "tokens_per_second": meta.get("performance", {}).get("tokens_per_second"),
                "elapsed_seconds": meta.get("performance", {}).get("elapsed_seconds"),
                "error": None,
                "text_column": meta.get("dataset", {}).get("text_column"),
                "label_column": meta.get("dataset", {}).get("labels", {}).get("label_column"),
            }
            records.append(record)
        except Exception as e:
            # Skip malformed files
            continue
    return pd.DataFrame(records)

print("Environment ready. Dark theme applied.")

Environment ready. Dark theme applied.


In [2]:
# =============================================================================
# CELL 2 – LOAD AND INSPECT DATASETS
# =============================================================================

from Get_Go_Emo import get_go
from Get_Isear import get_isr

goEmo = get_go()
isear = get_isr()

DATASETS = {
    "goEmo": goEmo,
    "ISEAR": isear,
}

for name, df in DATASETS.items():
    display_title(f"Dataset: {name}")
    display_info(f"Shape: {df.shape}")
    display(df.head(2))
    text_col = Extraction6.detect_text_column(df, show_verbose=False)
    label_cols = [c for c in df.columns if c.lower() in ("labels", "label", "emotion", "target")]
    label_col = label_cols[0] if label_cols else None
    display_info(f"Text column: <b>{text_col}</b><br>Label column: <b>{label_col or 'None'}</b><br>Samples: <b>{len(df):,}</b>")

display_title("Unified ID Scheme")
display_info("""
Each sample is assigned a unique integer ID (0..N-1) matching its row index.
This ID links:
• Input text (original DataFrame index)
• Hidden state vector (row in hidden_states.npy)
• Label (row in labels.npy)
No shuffling occurs, guaranteeing one‑to‑one mapping.
""")

,labels,clean_text
0,[27],my favourite food is anything i didnt have to ...
1,[27],"now if he does off himself, everyone will thin..."


,clean_text,labels
0,during the period of falling in love each time...,1
1,when i was involved in a traffic accident,2


In [3]:
# =============================================================================
# CELL 3 – RUN MODEL MATRIX (ENHANCED, WITH SAFEGUARDS)
# =============================================================================

# This call will automatically resume from existing completions.
# If interrupted, simply re-run this cell to continue.
results = Extraction6.run_model_matrix(
    datasets=DATASETS,
    groups=None,                     # all 25 models
    base_output="/Volumes/Amirali/hidden_states",
    pooling="mean",
    max_length=512,
    use_half_precision=True,
    auto_batch_size=False,
    flush_every_batches=8,
    continue_on_model_error=True,
    show_verbose=SHOW_VERBOSE,
    show_info=SHOW_INFO,
    show_critical=SHOW_CRITICAL,
    show_debug=SHOW_DEBUG,
)

print(f"Returned result records: {len(results)}")


╔══════════════════════════════════════════════════════════════════════════════════════╗
║ MODEL MATRIX                                                                         ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
  Requested experiment : baseline_v5_001
  Transformers         : 4.47.1
  PyTorch              : 2.2.2
  Models               : 25
  Datasets             : 2
  Runtime batch tuning : DISABLED
  Output root          : /Volumes/Amirali/hidden_states
01. [01_encoders] BERT         0.11B  google-bert/bert-base-uncased
02. [01_encoders] DistilBERT   0.066B  distilbert/distilbert-base-uncased
03. [01_encoders] RoBERTa      0.125B  FacebookAI/roberta-base
04. [01_encoders] ELECTRA      0.014B  google/electra-small-discriminator
05. [01_encoders] DeBERTa      0.14B  microsoft/deberta-v3-small
06. [02_early_decoders] GPT          0.124B  gpt2
07. [02_early_decoders] GPT-Neo      0.125B  EleutherAI/gpt-neo-125m
08. [02_early_decode

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  ✓ Snapshot ready
    Attempt time        : 1.14s
    Total preparation   : 1.17s
    Checkpoint size     : 0.410 GiB
    Snapshot            : /Volumes/Amirali/hidden_states/huggingface_cache/hub/models--google-bert--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594

────────────────────────────────────────────────────────────────────────────────────────
MODEL PREFLIGHT
────────────────────────────────────────────────────────────────────────────────────────
  model                             : google-bert/bert-base-uncased
  model_type                        : bert
  architecture                      : encoder
  dtype                             : torch.float32
  device                            : cpu
  layers                            : 12
  hidden_size                       : 768
  hidden_states                     : 13
  max_length                        : 512
  checkpoint                        : {'active_bytes': 440453864, 'active_gb': 0.4102046266198158, '

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

  ✓ Snapshot ready
    Attempt time        : 0.61s
    Total preparation   : 0.64s
    Checkpoint size     : 0.266 GiB
    Snapshot            : /Volumes/Amirali/hidden_states/huggingface_cache/hub/models--microsoft--deberta-v3-small/snapshots/a36c739020e01763fe789b4b85e2df55d6180012

────────────────────────────────────────────────────────────────────────────────────────
MODEL PREFLIGHT
────────────────────────────────────────────────────────────────────────────────────────
  model                             : microsoft/deberta-v3-small
  model_type                        : deberta-v2
  architecture                      : encoder
  dtype                             : torch.float32
  device                            : cpu
  layers                            : 6
  hidden_size                       : 768
  hidden_states                     : 7
  max_length                        : 512
  checkpoint                        : {'active_bytes': 286063365, 'active_gb': 0.26641726959496737, 'i

...

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

  ✓ Snapshot ready
    Attempt time        : 1.75s
    Total preparation   : 1.76s
    Checkpoint size     : 0.920 GiB
    Snapshot            : /Volumes/Amirali/hidden_states/huggingface_cache/hub/models--Qwen--Qwen2-0.5B/snapshots/91d2aff3f957f99e4c74c962f2f408dcc88a18d8

────────────────────────────────────────────────────────────────────────────────────────
MODEL PREFLIGHT
────────────────────────────────────────────────────────────────────────────────────────
  model                             : Qwen/Qwen2-0.5B
  model_type                        : qwen2
  architecture                      : decoder
  dtype                             : torch.float32
  device                            : cpu
  layers                            : 24
  hidden_size                       : 896
  hidden_states                     : 25
  max_length                        : 512
  checkpoint                        : {'active_bytes': 988097824, 'active_gb': 0.920237809419632, 'inventory': {'safetensors': 

goEmo:   0%|          | 32/54263 [00:00<?, ?sample/s]

[start] starting at absolute sample 32
[batch] 32:64 processed
[batch] 64:96 processed
[batch] 96:128 processed
[batch] 128:160 processed

────────────────────────────────────────────────────────────────────────────────────────
RUNTIME / SPEED TELEMETRY :: goEmo
────────────────────────────────────────────────────────────────────────────────────────
  Progress             : 160/54,263 (0.29%)
  Newly computed       : 128
  Wall time            : 15.34s
  Throughput           : 8.34 samples/s
  ETA                  : 1h 48m 05.1s
  Current stage        : measurement

  LAST BATCH
    Range              : 128:160
    New samples        : 32
    Total              : 3.81s
    Forward            : 3.51s
    Tokenization       : 0.01s
    Input transfer     : 0.00s
    Pooling            : 0.02s
    Conversion         : 0.00s
    Memmap write       : 0.13s
    Flush              : 0.06s
    Sequence length    : 33
    Tokens             : 513
    Token throughput   : 134.47 tokens/s

  ROLL

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# CELL 4 – COLLECT ALL RESULTS FROM DISK (PARTIAL INCLUDED)
# =============================================================================

exp_root = Path("/Volumes/Amirali/hidden_states/experiments/baseline_v5_001")
df_results = load_experiment_results(exp_root)

if not df_results.empty:
    display_title("Extraction Progress")
    display(df_results[['model', 'dataset', 'status', 'completed_samples', 'total_samples']].head(15))
    display_info(f"Total dataset‑model pairs found: <b>{len(df_results)}</b>")
else:
    display_info("No extraction metadata found yet. Run the extraction first (Cell 3).")

In [ ]:
# =============================================================================
# CELL 5 – VISUALISATIONS & ANALYSIS
# =============================================================================

if not df_results.empty:
    # Prepare data for plotting
    df = df_results.copy()
    df['completion_pct'] = df['completed_samples'] / df['total_samples'] * 100
    df['status_clean'] = df['status'].replace({'complete': 'Complete', 'partial': 'Partial', 'failed': 'Failed', 'already_complete': 'Already Complete'})

    sns.set_style("darkgrid")
    plt.rcParams.update({
        'figure.facecolor': '#1e1e1e',
        'axes.facecolor': '#2d2d2d',
        'axes.edgecolor': '#d4d4d4',
        'axes.labelcolor': '#d4d4d4',
        'text.color': '#d4d4d4',
        'xtick.color': '#d4d4d4',
        'ytick.color': '#d4d4d4',
        'grid.color': '#444444',
        'legend.facecolor': '#2d2d2d',
        'legend.edgecolor': '#d4d4d4',
    })

    # ---- 1. Completion status per model/dataset ----
    fig, ax = plt.subplots(figsize=(12, 8))
    pivot = df.pivot_table(index='model', columns='dataset', values='completion_pct', aggfunc='max')
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="viridis", cbar_kws={'label': 'Completion %'}, ax=ax)
    ax.set_title('Completion Percentage by Model and Dataset', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # ---- 2. Throughput (samples/sec) by model ----
    df_complete = df[df['samples_per_second'].notna()]
    if not df_complete.empty:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.barplot(data=df_complete, x='model', y='samples_per_second', hue='dataset', palette='coolwarm', ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_title('Extraction Throughput (samples/sec)', color='#4fc3f7')
        plt.tight_layout()
        plt.show()

    # ---- 3. Total time per model ----
    df_time = df.groupby('model')['elapsed_seconds'].sum().reset_index().sort_values('elapsed_seconds', ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=df_time, x='elapsed_seconds', y='model', palette='magma', ax=ax)
    ax.set_xlabel('Total Elapsed Time (seconds)')
    ax.set_title('Cumulative Extraction Time per Model', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # ---- 4. Label coverage for completed datasets ----
    # We'll just display label column info
    display_title("Label Columns Used")
    display(df[['model', 'dataset', 'label_column']].drop_duplicates())
else:
    display_info("No data to visualise. Please run extraction first.")

In [ ]:
# =============================================================================
# CELL 6 – FINAL SUMMARY & EXPORT
# =============================================================================

if not df_results.empty:
    # Save consolidated CSV
    output_csv = exp_root / "extraction_summary.csv"
    df_results.to_csv(output_csv, index=False)
    display_title("Final Report")
    display(df_results)
    display_info(f"Report exported to <code>{output_csv}</code>")
else:
    display_info("Nothing to export yet.")